In [1]:
import fitz
import re
from pathlib import Path 
import pymupdf4llm


In [ ]:
input_dir = Path("../../data/raw")
output_dir = Path("../../data/processed")

In [3]:
    
for pdf in input_dir.glob("*.pdf"):
    md_text = pymupdf4llm.to_markdown(str(pdf))

    output_file = output_dir / f"{pdf.stem}.md"

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(md_text)

    print(f"Converted {pdf.name}")

Converted deepseek.pdf
Converted toolformer.pdf
Converted llava.pdf
Converted gpt.pdf
Converted attention.pdf
Converted sentence_bert.pdf
Converted dspy.pdf
Converted beir.pdf
Converted hyde.pdf
Converted lora.pdf
Converted rag.pdf
Converted clip.pdf
Converted paged_attention.pdf
Converted qlora.pdf
Converted self_rag.pdf
Converted qwen2.pdf
Converted bert.pdf
Converted longformer.pdf
Converted crag.pdf
Converted colbert.pdf
Converted colbert2.pdf
Converted react.pdf
Converted realm.pdf
Converted kv_cache.pdf
Converted dense_retireval.pdf
Converted flas_attention.pdf
Converted blip.pdf


In [24]:
def clean_markdown(text):

    # remove image placeholders
    text = re.sub(
        r"==>\s*picture\s*\[.*?\]\s*intentionally omitted\s*<==",
        "",
        text,
        flags=re.IGNORECASE
    )

    # remove references / acknowledgements
    stop_headers = [
    r"^##\s+(?:\*\*)?references(?:\*\*)?",
    r"^##\s+(?:\*\*)?bibliography(?:\*\*)?",
    r"^##\s+(?:\*\*)?acknowledg(?:ement|ements)?(?:\*\*)?",
]

    lines = text.split("\n")

    cleaned_lines = []

    for line in lines:

        stop = False

        for pattern in stop_headers:
            if re.match(pattern, line.strip(), re.IGNORECASE):
                stop = True
                break

        if stop:
            break

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # remove excessive whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)

    # merge broken sentence linebreaks
    text = re.sub(r"(?<!\n)\n(?!#|\n)", " ", text)

    # collapse spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [25]:
md_file = output_dir / "attention.md"\

with open(md_file, "r", encoding="utf-8") as f:
    md_text = f.read()

cleaned_text = clean_markdown(md_text)
cleaned_text

'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. \n\n# **Attention Is All You Need** \n\n**Ashish Vaswani** _[∗]_ **Noam Shazeer** _[∗]_ **Niki Parmar** _[∗]_ **Jakob Uszkoreit** _[∗]_ Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com` \n\n**Llion Jones** _[∗]_ **Aidan N. Gomez** _[∗†]_ **Łukasz Kaiser** _[∗]_ Google Research University of Toronto Google Brain `llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com` \n\n**Illia Polosukhin** _[∗‡]_ \n\n``` illia.polosukhin@gmail.com ```\n\n## **Abstract** \n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network arc

In [26]:
def parse_sections(md_text):

    lines = md_text.split("\n")

    nodes = []

    current_section = None
    current_subsection = None

    buffer = []

    KNOWN_HEADERS = {
        "abstract",
        "introduction",
        "background",
        "related work",
        "method",
        "methods",
        "approach",
        "model architecture",
        "training",
        "experiments",
        "results",
        "evaluation",
        "discussion",
        "limitations",
        "conclusion",
        "references",
        "appendix"
    }

    def flush_buffer():
        nonlocal buffer

        content = "\n".join(buffer).strip()

        # don't create empty chunks
        if content and len(content.split()) > 20:

            nodes.append({
                "section": current_section,
                "subsection": current_subsection,
                "text": content
            })

        buffer = []

    for line in lines:

        line = line.strip()

        # markdown header
        match = re.match(
            r"^(#{1,6})\s+(.+)$",
            line
        )

        if match:

            level = len(match.group(1))
            title = match.group(2).strip()

            # remove markdown bold
            title = re.sub(r"\*\*", "", title)
            title = re.sub(r"\s+", " ", title)

            title_lower = title.lower()

            is_valid_header = False

            numbering_match = re.match(
                r"^(\d+(?:\.\d+)*)\s+(.+)$",
                title
            )

            # ---------- NUMBERED HEADERS ----------
            if numbering_match:

                numbering = numbering_match.group(1)
                depth = numbering.count(".")

                is_valid_header = True

                flush_buffer()

                # 3 Title
                if depth == 0:

                    current_section = title
                    current_subsection = None

                # 3.1 Subtitle
                elif depth == 1:

                    current_subsection = title

                # 3.2.1 Child subsection
                elif depth >= 2:

                    # keep under same subsection
                    # append title into body for context
                    buffer.append(f"\n{title}\n")

                continue

            # ---------- SEMANTIC HEADERS ----------
            for known in KNOWN_HEADERS:
                if known in title_lower:

                    is_valid_header = True

                    flush_buffer()

                    current_section = title
                    current_subsection = None
                    break

            if is_valid_header:
                continue

        # normal content
        buffer.append(line)

    flush_buffer()

    return nodes

In [27]:
nodes = parse_sections(cleaned_text)
nodes[:5]

[{'section': None,
  'subsection': None,
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n# **Attention Is All You Need**\n\n**Ashish Vaswani** _[∗]_ **Noam Shazeer** _[∗]_ **Niki Parmar** _[∗]_ **Jakob Uszkoreit** _[∗]_ Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com`\n\n**Llion Jones** _[∗]_ **Aidan N. Gomez** _[∗†]_ **Łukasz Kaiser** _[∗]_ Google Research University of Toronto Google Brain `llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com`\n\n**Illia Polosukhin** _[∗‡]_\n\n``` illia.polosukhin@gmail.com ```'},
 {'section': 'Abstract',
  'subsection': None,
  'text': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the enc

In [28]:
import configparser
from transformers import AutoTokenizer

# ---------------- CONFIG ---------------- #
config = configparser.ConfigParser()
config.read('constants.ini')


MODEL_NAME = config.get('CHUNK', 'MODEL_NAME')
CHUNK_SIZE = config.getint('CHUNK', 'CHUNK_SIZE')
OVERLAP = config.getint('CHUNK', 'OVERLAP')
MIN_SUBSECTION_TOKENS = config.getint('CHUNK', 'MIN_SUBSECTION_TOKENS')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

In [29]:
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_tokens(text):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + CHUNK_SIZE

        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append(chunk_text)

        start += CHUNK_SIZE - OVERLAP

    return chunks


def merge_small_subsections(nodes):

    merged = []

    buffer = None

    for node in nodes:

        token_count = count_tokens(node["text"])
        print(f"Section: {node['section']} | Subsection: {node['subsection']} | Tokens: {token_count}")
        if token_count >= MIN_SUBSECTION_TOKENS:

            if buffer:
                merged.append(buffer)
                buffer = None

            merged.append(node)

        else:

            if buffer is None:

                buffer = node.copy()

            else:

                # only merge within same section
                if (
                    buffer["section"] ==
                    node["section"]
                ):

                    buffer["text"] += "\n\n" + node["text"]

                else:

                    merged.append(buffer)
                    buffer = node.copy()

    if buffer:
        merged.append(buffer)

    return merged

In [30]:
sections = merge_small_subsections(nodes)
sections[:5]

Section: None | Subsection: None | Tokens: 204
Section: Abstract | Subsection: None | Tokens: 443
Section: 1 Introduction | Subsection: None | Tokens: 357
Section: 2 Background | Subsection: None | Tokens: 351
Section: 3 Model Architecture | Subsection: None | Tokens: 177
Section: 3 Model Architecture | Subsection: 3.1 Encoder and Decoder Stacks | Tokens: 296
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 141
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 331
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 445
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 251
Section: 3 Model Architecture | Subsection: 3.3 Position-wise Feed-Forward Networks | Tokens: 160
Section: 3 Model Architecture | Subsection: 3.4 Embeddings and Softmax | Tokens: 311
Section: 3 Model Architecture | Subsection: 3.5 Positional Encoding | Tokens: 292
Section: 4 Why Self-Attention | Subsection: None | Tokens: 674
Section: 5 Training | 

[{'section': None,
  'subsection': None,
  'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n# **Attention Is All You Need**\n\n**Ashish Vaswani** _[∗]_ **Noam Shazeer** _[∗]_ **Niki Parmar** _[∗]_ **Jakob Uszkoreit** _[∗]_ Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com`\n\n**Llion Jones** _[∗]_ **Aidan N. Gomez** _[∗†]_ **Łukasz Kaiser** _[∗]_ Google Research University of Toronto Google Brain `llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com`\n\n**Illia Polosukhin** _[∗‡]_\n\n``` illia.polosukhin@gmail.com ```'},
 {'section': 'Abstract',
  'subsection': None,
  'text': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the enc

In [31]:
for section in sections:
    print(f"Section: {section['section']} | Subsection: {section['subsection']} | Tokens: {count_tokens(section['text'])}")

Section: None | Subsection: None | Tokens: 204
Section: Abstract | Subsection: None | Tokens: 443
Section: 1 Introduction | Subsection: None | Tokens: 357
Section: 2 Background | Subsection: None | Tokens: 351
Section: 3 Model Architecture | Subsection: None | Tokens: 177
Section: 3 Model Architecture | Subsection: 3.1 Encoder and Decoder Stacks | Tokens: 296
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 141
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 331
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 445
Section: 3 Model Architecture | Subsection: 3.2 Attention | Tokens: 251
Section: 3 Model Architecture | Subsection: 3.3 Position-wise Feed-Forward Networks | Tokens: 160
Section: 3 Model Architecture | Subsection: 3.4 Embeddings and Softmax | Tokens: 311
Section: 3 Model Architecture | Subsection: 3.5 Positional Encoding | Tokens: 292
Section: 4 Why Self-Attention | Subsection: None | Tokens: 674
Section: 5 Training | 